# Viral Prediction with BiLSTM + Tabular\n
\n
This notebook is designed for Google Colab.\n
It trains a binary classifier with text (BiLSTM) + numeric/categorical features.

In [ ]:
!pip -q install torch pandas numpy scikit-learn transformers

In [ ]:
from pathlib import Path\n
import os\n
import random\n
\n
import numpy as np\n
import pandas as pd\n
import torch\n
import torch.nn as nn\n
from sklearn.metrics import classification_report, roc_auc_score\n
from sklearn.model_selection import train_test_split\n
from sklearn.preprocessing import StandardScaler\n
from torch.utils.data import DataLoader, Dataset\n
from transformers import AutoTokenizer\n
\n
print('Torch:', torch.__version__)\n
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# Optional: mount Google Drive if your data is there\n
# from google.colab import drive\n
# drive.mount('/content/drive')\n
\n
TARGET_COLUMN = 'viral'\n
TEXT_COLUMN = 'text'\n
RANDOM_STATE = 42\n
\n
NUMERIC_COLUMNS = [\n
    'upload_hour',\n
    'is_weekend',\n
    'duration_sec',\n
    'creator_avg_views',\n
    'title_length',\n
    'has_emoji',\n
]\n
\n
CATEGORICAL_COLUMNS = [\n
    'category', 'genre', 'sound_type', 'music_track',\n
    'publish_dayofweek', 'publish_period', 'season', 'event_season',\n
    'creator_tier', 'platform', 'country', 'region', 'language', 'traffic_source',\n
]\n
\n
# TODO: set this path to your processed csv\n
DATA_PATH = '/content/project_viral/data/processed/processed_data_20000_0.06.csv'\n
\n
CONFIG = {\n
    'test_size': 0.2,\n
    'batch_size': 64,\n
    'epochs': 10,\n
    'learning_rate': 1e-3,\n
    'dropout': 0.3,\n
    'threshold': 0.5,\n
    'tokenizer_name': 'distilbert-base-uncased',\n
    'max_length': 64,\n
    'embed_dim': 128,\n
    'lstm_hidden_dim': 128,\n
    'lstm_layers': 1,\n
    'bidirectional': True,\n
    'token_mixer': 'projected_concat',\n
    'proj_dim': 64,\n
    'classifier_hidden_dims': [128, 64],\n
}\n
\n
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n
print('Using device:', DEVICE)

In [ ]:
def set_seed(seed: int):\n
    random.seed(seed)\n
    np.random.seed(seed)\n
    torch.manual_seed(seed)\n
    if torch.cuda.is_available():\n
        torch.cuda.manual_seed(seed)\n
        torch.cuda.manual_seed_all(seed)\n
    torch.backends.cudnn.deterministic = True\n
    torch.backends.cudnn.benchmark = False\n
\n
\n
class ViralDataset(Dataset):\n
    def __init__(self, input_ids, attention_mask, num_features, cat_features, labels):\n
        self.input_ids = torch.tensor(input_ids, dtype=torch.long)\n
        self.attention_mask = torch.tensor(attention_mask, dtype=torch.long)\n
        self.num_features = torch.tensor(num_features, dtype=torch.float32)\n
        self.cat_features = torch.tensor(cat_features, dtype=torch.float32)\n
        self.labels = torch.tensor(labels, dtype=torch.float32)\n
\n
    def __len__(self):\n
        return len(self.labels)\n
\n
    def __getitem__(self, idx):\n
        return (\n
            self.input_ids[idx],\n
            self.attention_mask[idx],\n
            self.num_features[idx],\n
            self.cat_features[idx],\n
            self.labels[idx],\n
        )\n
\n
\n
class TokenMixer(nn.Module):\n
    def __init__(self, text_dim, num_dim, cat_dim, token_mixer, proj_dim, dropout):\n
        super().__init__()\n
        self.token_mixer = token_mixer\n
\n
        if token_mixer == 'raw_concat':\n
            self.output_dim = text_dim + num_dim + cat_dim\n
        elif token_mixer == 'projected_concat':\n
            self.text_proj = nn.Sequential(nn.Linear(text_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))\n
            self.num_proj = nn.Sequential(nn.Linear(num_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))\n
            self.cat_proj = nn.Sequential(nn.Linear(cat_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))\n
            self.output_dim = proj_dim * 3\n
        elif token_mixer == 'weighted_sum':\n
            self.text_proj = nn.Sequential(nn.Linear(text_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))\n
            self.num_proj = nn.Sequential(nn.Linear(num_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))\n
            self.cat_proj = nn.Sequential(nn.Linear(cat_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))\n
            self.text_gate = nn.Linear(proj_dim, 1)\n
            self.num_gate = nn.Linear(proj_dim, 1)\n
            self.cat_gate = nn.Linear(proj_dim, 1)\n
            self.output_dim = proj_dim\n
        elif token_mixer == 'attention_pool':\n
            self.text_proj = nn.Sequential(nn.Linear(text_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))\n
            self.num_proj = nn.Sequential(nn.Linear(num_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))\n
            self.cat_proj = nn.Sequential(nn.Linear(cat_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))\n
            self.attention = nn.MultiheadAttention(embed_dim=proj_dim, num_heads=1, batch_first=True, dropout=dropout)\n
            self.output_dim = proj_dim\n
        else:\n
            raise ValueError(f'Unsupported token_mixer: {token_mixer}')\n
\n
    def forward(self, x_text, x_num, x_cat):\n
        if self.token_mixer == 'raw_concat':\n
            return torch.cat([x_text, x_num, x_cat], dim=1)\n
\n
        if self.token_mixer == 'projected_concat':\n
            return torch.cat([self.text_proj(x_text), self.num_proj(x_num), self.cat_proj(x_cat)], dim=1)\n
\n
        if self.token_mixer == 'weighted_sum':\n
            e_text = self.text_proj(x_text)\n
            e_num = self.num_proj(x_num)\n
            e_cat = self.cat_proj(x_cat)\n
            gates = torch.cat([self.text_gate(e_text), self.num_gate(e_num), self.cat_gate(e_cat)], dim=1)\n
            w = torch.softmax(gates, dim=1)\n
            return w[:, 0:1] * e_text + w[:, 1:2] * e_num + w[:, 2:3] * e_cat\n
\n
        if self.token_mixer == 'attention_pool':\n
            e_text = self.text_proj(x_text)\n
            e_num = self.num_proj(x_num)\n
            e_cat = self.cat_proj(x_cat)\n
            tokens = torch.stack([e_text, e_num, e_cat], dim=1)\n
            attn_out, _ = self.attention(tokens, tokens, tokens)\n
            return attn_out.mean(dim=1)\n
\n
        raise ValueError('Invalid token_mixer')\n
\n
\n
class BiLSTMTabularClassifier(nn.Module):\n
    def __init__(\n
        self,\n
        vocab_size,\n
        pad_token_id,\n
        embed_dim,\n
        lstm_hidden_dim,\n
        lstm_layers,\n
        bidirectional,\n
        num_dim,\n
        cat_dim,\n
        token_mixer,\n
        proj_dim,\n
        dropout,\n
        classifier_hidden_dims,\n
    ):\n
        super().__init__()\n
        self.bidirectional = bidirectional\n
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_token_id)\n
        self.lstm = nn.LSTM(\n
            input_size=embed_dim,\n
            hidden_size=lstm_hidden_dim,\n
            num_layers=lstm_layers,\n
            batch_first=True,\n
            dropout=dropout if lstm_layers > 1 else 0.0,\n
            bidirectional=bidirectional,\n
        )\n
\n
        text_dim = lstm_hidden_dim * (2 if bidirectional else 1)\n
        self.token_mixer = TokenMixer(text_dim, num_dim, cat_dim, token_mixer, proj_dim, dropout)\n
\n
        layers = []\n
        in_dim = self.token_mixer.output_dim\n
        for h in classifier_hidden_dims:\n
            layers.extend([nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)])\n
            in_dim = h\n
        layers.append(nn.Linear(in_dim, 1))\n
        self.classifier = nn.Sequential(*layers)\n
\n
    def _encode_text(self, input_ids, attention_mask):\n
        lengths = attention_mask.sum(dim=1).cpu()\n
        emb = self.embedding(input_ids)\n
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)\n
        _, (h_n, _) = self.lstm(packed)\n
\n
        if self.bidirectional:\n
            return torch.cat([h_n[-2], h_n[-1]], dim=1)\n
        return h_n[-1]\n
\n
    def forward(self, input_ids, attention_mask, x_num, x_cat):\n
        text_repr = self._encode_text(input_ids, attention_mask)\n
        mixed = self.token_mixer(text_repr, x_num, x_cat)\n
        return self.classifier(mixed).squeeze(1)\n
\n
\n
def prepare_tabular_features(train_df, test_df):\n
    scaler = StandardScaler()\n
    train_num = scaler.fit_transform(train_df[NUMERIC_COLUMNS]).astype(np.float32)\n
    test_num = scaler.transform(test_df[NUMERIC_COLUMNS]).astype(np.float32)\n
\n
    train_cat = pd.get_dummies(train_df[CATEGORICAL_COLUMNS], drop_first=False)\n
    test_cat = pd.get_dummies(test_df[CATEGORICAL_COLUMNS], drop_first=False)\n
    test_cat = test_cat.reindex(columns=train_cat.columns, fill_value=0)\n
\n
    return train_num, test_num, train_cat.to_numpy(np.float32), test_cat.to_numpy(np.float32)\n
\n
\n
def build_dataloaders(train_df, test_df, tokenizer, max_length, batch_size):\n
    train_num, test_num, train_cat, test_cat = prepare_tabular_features(train_df, test_df)\n
\n
    train_enc = tokenizer(\n
        train_df[TEXT_COLUMN].tolist(),\n
        padding='max_length', truncation=True, max_length=max_length, return_tensors='np'\n
    )\n
    test_enc = tokenizer(\n
        test_df[TEXT_COLUMN].tolist(),\n
        padding='max_length', truncation=True, max_length=max_length, return_tensors='np'\n
    )\n
\n
    y_train = train_df[TARGET_COLUMN].to_numpy(dtype=np.float32)\n
    y_test = test_df[TARGET_COLUMN].to_numpy(dtype=np.float32)\n
\n
    train_ds = ViralDataset(train_enc['input_ids'], train_enc['attention_mask'], train_num, train_cat, y_train)\n
    test_ds = ViralDataset(test_enc['input_ids'], test_enc['attention_mask'], test_num, test_cat, y_test)\n
\n
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)\n
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)\n
    return train_loader, test_loader, y_train, y_test, train_num.shape[1], train_cat.shape[1]\n
\n
\n
def evaluate(model, dataloader, criterion):\n
    model.eval()\n
    total_loss = 0.0\n
    all_probs, all_labels = [], []\n
\n
    with torch.no_grad():\n
        for input_ids, attention_mask, x_num, x_cat, labels in dataloader:\n
            input_ids = input_ids.to(DEVICE)\n
            attention_mask = attention_mask.to(DEVICE)\n
            x_num = x_num.to(DEVICE)\n
            x_cat = x_cat.to(DEVICE)\n
            labels = labels.to(DEVICE)\n
\n
            logits = model(input_ids, attention_mask, x_num, x_cat)\n
            loss = criterion(logits, labels)\n
            probs = torch.sigmoid(logits)\n
\n
            total_loss += loss.item() * input_ids.size(0)\n
            all_probs.extend(probs.cpu().numpy())\n
            all_labels.extend(labels.cpu().numpy())\n
\n
    return total_loss / len(dataloader.dataset), np.array(all_probs), np.array(all_labels)

In [ ]:
set_seed(RANDOM_STATE)\n
\n
df = pd.read_csv(DATA_PATH)\n
print('Dataset shape:', df.shape)\n
train_df, test_df = train_test_split(\n
    df,\n
    test_size=CONFIG['test_size'],\n
    random_state=RANDOM_STATE,\n
    stratify=df[TARGET_COLUMN],\n
)\n
print('Train size:', len(train_df), '| Test size:', len(test_df))\n
\n
tokenizer = AutoTokenizer.from_pretrained(CONFIG['tokenizer_name'])\n
train_loader, test_loader, y_train, y_test, num_dim, cat_dim = build_dataloaders(\n
    train_df, test_df, tokenizer, CONFIG['max_length'], CONFIG['batch_size']\n
)\n
\n
model = BiLSTMTabularClassifier(\n
    vocab_size=tokenizer.vocab_size,\n
    pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0,\n
    embed_dim=CONFIG['embed_dim'],\n
    lstm_hidden_dim=CONFIG['lstm_hidden_dim'],\n
    lstm_layers=CONFIG['lstm_layers'],\n
    bidirectional=CONFIG['bidirectional'],\n
    num_dim=num_dim,\n
    cat_dim=cat_dim,\n
    token_mixer=CONFIG['token_mixer'],\n
    proj_dim=CONFIG['proj_dim'],\n
    dropout=CONFIG['dropout'],\n
    classifier_hidden_dims=CONFIG['classifier_hidden_dims'],\n
).to(DEVICE)\n
\n
positive_count = float(y_train.sum())\n
negative_count = float(len(y_train) - positive_count)\n
pos_weight = torch.tensor([negative_count / max(positive_count, 1.0)], dtype=torch.float32).to(DEVICE)\n
\n
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)\n
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])\n
\n
best_auc = float('-inf')\n
best_state = None\n
\n
for epoch in range(CONFIG['epochs']):\n
    model.train()\n
    total_train_loss = 0.0\n
\n
    for input_ids, attention_mask, x_num, x_cat, labels in train_loader:\n
        input_ids = input_ids.to(DEVICE)\n
        attention_mask = attention_mask.to(DEVICE)\n
        x_num = x_num.to(DEVICE)\n
        x_cat = x_cat.to(DEVICE)\n
        labels = labels.to(DEVICE)\n
\n
        optimizer.zero_grad()\n
        logits = model(input_ids, attention_mask, x_num, x_cat)\n
        loss = criterion(logits, labels)\n
        loss.backward()\n
        optimizer.step()\n
\n
        total_train_loss += loss.item() * input_ids.size(0)\n
\n
    train_loss = total_train_loss / len(train_loader.dataset)\n
    test_loss, test_probs, test_labels = evaluate(model, test_loader, criterion)\n
    test_auc = roc_auc_score(test_labels, test_probs)\n
\n
    if test_auc > best_auc:\n
        best_auc = test_auc\n
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n
\n
    print(f'Epoch {epoch + 1}/{CONFIG["epochs"]} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | Test ROC-AUC: {test_auc:.4f}')\n
\n
if best_state is not None:\n
    model.load_state_dict(best_state)\n
\n
test_loss, test_probs, test_labels = evaluate(model, test_loader, criterion)\n
print('Best ROC-AUC during training:', round(best_auc, 4))\n
print('Final Test Loss:', round(test_loss, 4))\n
print('Final Test ROC-AUC:', round(roc_auc_score(test_labels, test_probs), 4))\n
print('\\nClassification Report:\\n')\n
print(classification_report(test_labels, (test_probs >= CONFIG['threshold']).astype(int), digits=4))

In [ ]:
# Optional: save model checkpoint\n
save_path = '/content/bilstm_tabular_best.pt'\n
torch.save(model.state_dict(), save_path)\n
print('Saved model to:', save_path)